# Number of visits per pointing in each filter (u, g, r, i, z, y), per survey type, in baseline survey v5.3

This notebook complements `10_FilterBalance`. Instead of the fraction of visits per filter versus time, it shows the **distribution of the number of visits per pointing** in each of the six filters, separately for each survey category (same categories and same masks as in `10_FilterBalance`: `WFD`, `WFD low dust`, `DDF`, mini-surveys, micro-surveys).

- creation date: 2026-11-22
- companion notebooks: `10_FilterBalance` (filter balance versus time), `02_SurveyAreas`, `07_WFDMiniMicroS_NVisitsVsTime`
- survey modes definition: https://github.com/lsst/survey-strategy_lsst_io

## Goal and outputs

For each survey category, the number of visits received by each pointing is histogrammed, **one histogram per filter, with the usual filter colours** (same colour code as in `10_FilterBalance`). The legend gives, for each filter, the **mean** and the **median** number of visits per pointing.

**Figures** (one panel per category, same grid as the figures of `10_FilterBalance`):

1. **Figure 1**: non-stacked histograms (the six filters are overlaid);
2. **Figure 2**: stacked histograms (the six filters are stacked, the total height is the number of pointings).

**Steps**

1. Region labels (scheduler map), loading of the visits and boolean masks of the categories (same as `10_FilterBalance`).
2. Number of visits per pointing, per filter and per category.
3. Table of the mean and median number of visits per pointing (`.csv`).
4. Figures 1 and 2.

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (variable `opsdb`).

**Outputs**: the tables (`.csv`) go to `DATA10bis_FilterBalance/`; all figures (`.png` and `.pdf`) go to `FIG10bis_FilterBalance/`.

## 0. Setup

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

from rubin_scheduler.scheduler.utils import CurrentAreaMap

In [ ]:
plt.rcParams.update(
    {
        "font.size": 14,  # general size
        "axes.labelsize": 16,  # X and Y labels
        "axes.titlesize": 16,  # axes title
        "xtick.labelsize": 14,  # X ticks
        "ytick.labelsize": 14,  # Y ticks
        "legend.fontsize": 14,  # legend text
        "legend.title_fontsize": 14,
    }
)

In [ ]:
# Output directories: tables go to DATA_DIR, figures go to FIG_DIR
DATA_DIR = "DATA10bis_FilterBalance"
FIG_DIR = "FIG10bis_FilterBalance"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# OpSim database. Replace by another path to analyse another run.
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
opsim_name = os.path.split(opsdb)[-1].replace(".db", "")
print(opsdb)

BANDS = list("ugrizy")
# Usual Rubin colours of the six filters (same as in 10_FilterBalance)
BAND_COLORS = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}
NSIDE = 64  # HEALPix resolution of the scheduler map (region labels)

MIN_VISITS_TOTAL = 1000  # below this, the category is removed from the study

# Definition of a pointing (see section 2)
POINTING_MODE = "healpix"  # "healpix" or "radec"
NSIDE_POINTING = 64  # healpix mode: HEALPix resolution of the pointings
FOV_RADIUS_DEG = 1.75  # healpix mode: radius of the field of view (area 9.6 deg2)
RADEC_ROUND_DEG = 0.01  # radec mode: rounding of (fieldRA, fieldDec), in degrees

# Histograms
N_BINS = 50  # maximum number of bins of a histogram
LOG_Y = False  # True: logarithmic scale for the number of pointings

# Region definition: name -> list of labels of the scheduler map.
# 'euclid_overlap' is merged with 'lowdust' (as in 02_SurveyAreas and 07).
WFD_REGIONS = {
    "low dust": ["lowdust", "euclid_overlap"],
    "bulge": ["bulgy"],
    "virgo": ["virgo"],
    "SMC-LMC": ["LMC_SMC"],
}
MINI_SURVEYS = {
    "NES": ["nes"],
    "SCP": ["scp"],
    "MW non-WFD": ["dusty_plane"],
}

## 1. Region labels, loading of the visits and categories

Same code as in `10_FilterBalance`. `CurrentAreaMap` returns one survey-mode label per HEALPix pixel; a label of the map not listed in `WFD_REGIONS` or `MINI_SURVEYS` is automatically added as a mini-survey. The micro-surveys are selected from `scheduler_note` (priority ToO > TwilightNEO > RGES > DDF); the other visits are labelled by the pixel containing the visit centre.

Each visit is counted once in `WFD`, `DDF`, `Mini-surveys (total)`, `Micro-surveys without DDF (total)` (check below). `WFD low dust` is a subset of `WFD`, and `All visits` is the whole survey.

In [ ]:
# Scheduler footprint: one survey-mode label per HEALPix pixel
skymap = CurrentAreaMap(nside=NSIDE)
maps, labels = skymap.return_maps()
labels = np.asarray(labels).astype(str)
label_names = [l for l in np.unique(labels) if l != ""]
print("Labels in the target map:", label_names)

# Labels configured above but absent from the map
for name, labs in {**WFD_REGIONS, **MINI_SURVEYS}.items():
    for l in labs:
        if l not in label_names:
            print(f"WARNING: label {l} of region {name} is not in the target map")

# Labels of the map not configured above: added as mini-surveys
used_labels = {l for labs in list(WFD_REGIONS.values()) + list(MINI_SURVEYS.values()) for l in labs}
for l in label_names:
    if l not in used_labels:
        print(f"Label {l} is not in WFD_REGIONS or MINI_SURVEYS: added as a mini-survey")
        MINI_SURVEYS[l] = [l]

mini_names = list(MINI_SURVEYS)
wfd_labels = [l for labs in WFD_REGIONS.values() for l in labs]
mini_labels = [l for labs in MINI_SURVEYS.values() for l in labs]

In [ ]:
query = (
    "select observationStartMJD, night, band, fieldRA, fieldDec, "
    "visitExposureTime, scheduler_note from observations"
)
conn = sqlite3.connect(opsdb)
visits = pd.read_sql(query, conn)
conn.close()

visits = visits.sort_values("observationStartMJD").reset_index(drop=True)
N_ALL = len(visits)
print(f"{N_ALL} visits in the survey")

# Micro-surveys from scheduler_note (one tag per visit)
note = visits["scheduler_note"].fillna("")
is_too = note.str.contains("ToO").values
is_twilight = note.str.contains("twilight_near_sun").values
is_rges = note.str.contains("RGES").values
is_ddf = note.str.contains("DD").values & ~is_rges
visits["micro"] = np.select(
    [is_too, is_twilight, is_rges, is_ddf],
    ["ToO", "TwilightNEO", "RGES", "DDF"],
    default="",
)
is_micro = (visits["micro"] != "").values

# Label of the HEALPix pixel containing the visit centre (RA, Dec in degrees)
pix = hp.ang2pix(NSIDE, visits["fieldRA"].values, visits["fieldDec"].values, lonlat=True)
visits["label"] = labels[pix]

# Index of the filter (0..5); -1 if the filter is not in BANDS
band_idx = visits["band"].map({b: i for i, b in enumerate(BANDS)}).fillna(-1).astype(int).values
valid_band = band_idx >= 0
print(f"Visits in a filter outside {BANDS}: {(~valid_band).sum()}")
print(visits["band"].value_counts().to_string())

In [ ]:
sky_label = visits["label"].values
not_micro = ~is_micro

masks = {}
masks["All visits"] = np.ones(N_ALL, dtype=bool)
masks["WFD"] = not_micro & np.isin(sky_label, wfd_labels)
masks["WFD low dust"] = not_micro & np.isin(sky_label, WFD_REGIONS["low dust"])
masks["DDF"] = visits["micro"].values == "DDF"
masks["Mini-surveys (total)"] = not_micro & np.isin(sky_label, mini_labels)
for name, labs in MINI_SURVEYS.items():
    masks[name] = not_micro & np.isin(sky_label, labs)
for name in ["RGES", "TwilightNEO", "ToO"]:
    masks[name] = visits["micro"].values == name
masks["Micro-surveys without DDF (total)"] = is_micro & ~masks["DDF"]
masks["Unassigned"] = not_micro & ~masks["WFD"] & ~masks["Mini-surveys (total)"]

n_sum = sum(
    masks[n].sum()
    for n in ["WFD", "DDF", "Mini-surveys (total)", "Micro-surveys without DDF (total)", "Unassigned"]
)
print(f"WFD + DDF + mini + micro without DDF + unassigned = {n_sum}, all visits = {N_ALL}")
assert n_sum == N_ALL, "a visit is counted twice or not at all"
print(f'Unassigned visits: {masks["Unassigned"].sum()}')

# Display order of the categories
CATEGORIES = (
    ["All visits", "WFD", "WFD low dust", "DDF", "Mini-surveys (total)"]
    + mini_names
    + ["RGES", "TwilightNEO", "ToO", "Micro-surveys without DDF (total)"]
)
CATEGORIES = [c for c in CATEGORIES if c in masks]

# Categories with too few visits are removed from the study
n_vis = {c: int((masks[c] & valid_band).sum()) for c in CATEGORIES}
for c in CATEGORIES:
    if n_vis[c] < MIN_VISITS_TOTAL:
        print(f"Category removed (too few visits): {c}, {n_vis[c]} visits")
CATEGORIES = [c for c in CATEGORIES if n_vis[c] >= MIN_VISITS_TOTAL]
print("Categories studied:", CATEGORIES)

## 2. Number of visits per pointing

The OpSim database has one row per visit, so a **pointing** has to be defined before counting the visits it receives. Two definitions are available (`POINTING_MODE` in the configuration):

- `healpix` (default): a pointing is a HEALPix pixel (`NSIDE_POINTING`); the number of visits of a pixel is the number of visits whose field of view (disc of radius `FOV_RADIUS_DEG` = 1.75 deg around `fieldRA`, `fieldDec`) contains the pixel centre. This is the usual number of visits per sky position (the one of a `Count` metric on a HEALPix slicer). It is the default because the field centres of the baseline are dithered by the scheduler, so very few visits share exactly the same `(fieldRA, fieldDec)`; the number of distinct `(fieldRA, fieldDec)` printed below allows to check this.
- `radec`: a pointing is a unique `(fieldRA, fieldDec)` pair, rounded to `RADEC_ROUND_DEG`.

For a given category, the pointings are the pixels (or the pairs) that received **at least one visit of the category** in any filter. A pointing never observed in a given filter has 0 visit in this filter: it is included in the histogram of this filter (first bin), in the mean and in the median.

The counts are done with a KD-tree (`scipy.spatial.cKDTree`) on the unit sphere, one tree per category and per filter.

In [ ]:
n_unique = visits[["fieldRA", "fieldDec"]].round(2).drop_duplicates().shape[0]
print(f"{N_ALL} visits, {n_unique} distinct (fieldRA, fieldDec) rounded to 0.01 deg")

npix = hp.nside2npix(NSIDE_POINTING)
if POINTING_MODE == "healpix":
    POINTING_DESC = f"HEALPix pixel nside={NSIDE_POINTING}, visit radius {FOV_RADIUS_DEG} deg"
    pix_vec = np.array(hp.pix2vec(NSIDE_POINTING, np.arange(npix))).T  # pixel centres, unit vectors
    vis_vec = hp.ang2vec(visits["fieldRA"].values, visits["fieldDec"].values, lonlat=True)
    chord = 2 * np.sin(np.radians(FOV_RADIUS_DEG) / 2)  # angular radius -> chord on the unit sphere
elif POINTING_MODE == "radec":
    POINTING_DESC = f"unique (fieldRA, fieldDec) rounded to {RADEC_ROUND_DEG} deg"
else:
    raise ValueError(f"unknown POINTING_MODE: {POINTING_MODE}")
print("Pointing definition:", POINTING_DESC)


def visits_per_pointing(sel):
    # Array (n_pointings, 6): number of visits of the selection in each filter, for each pointing
    sel = sel & valid_band
    if POINTING_MODE == "healpix":
        out = np.zeros((npix, len(BANDS)), dtype=int)
        for i in range(len(BANDS)):
            s = sel & (band_idx == i)
            if s.sum() == 0:
                continue
            tree = cKDTree(vis_vec[s])
            out[:, i] = tree.query_ball_point(pix_vec, chord, return_length=True, workers=-1)
        return out[out.sum(axis=1) > 0]  # keep the pointings with at least one visit
    # radec mode
    step = RADEC_ROUND_DEG
    df = pd.DataFrame(
        {
            "ra": np.round(visits["fieldRA"].values[sel] / step).astype(np.int64),
            "dec": np.round(visits["fieldDec"].values[sel] / step).astype(np.int64),
            "band": band_idx[sel],
        }
    )
    tab = df.groupby(["ra", "dec", "band"]).size().unstack("band", fill_value=0)
    return tab.reindex(columns=range(len(BANDS)), fill_value=0).values


vpp = {}  # category -> array (n_pointings, 6)
for cat in CATEGORIES:
    vpp[cat] = visits_per_pointing(masks[cat])
    print(
        f"{cat:35s} {len(vpp[cat]):8d} pointings, {vpp[cat].sum(axis=1).mean():8.1f} visits per pointing (all filters)"
    )

## 3. Mean and median number of visits per pointing

Table of the numbers given in the legends of the figures: number of pointings of the category, then mean and median number of visits per pointing in each filter. It is saved in `DATA10bis_FilterBalance/` (`..._visits_per_pointing_stats.csv`).

Caution: in `healpix` mode a visit is counted in all the pixels of its field of view (about 11 pixels at `nside=64`), so the numbers are visits per **sky position**, not visits per field centre.

In [ ]:
rows = {}
for cat in CATEGORIES:
    v = vpp[cat]
    row = {"N pointings": len(v)}
    for i, b in enumerate(BANDS):
        row[f"mean_{b}"] = v[:, i].mean()
    for i, b in enumerate(BANDS):
        row[f"median_{b}"] = np.median(v[:, i])
    rows[cat] = row
stats_table = pd.DataFrame(rows).T
stats_table["N pointings"] = stats_table["N pointings"].astype(int)
stats_table.round(2).to_csv(os.path.join(DATA_DIR, f"{opsim_name}_visits_per_pointing_stats.csv"))

fmt = {"N pointings": "{:,.0f}"}
fmt.update({c: "{:.1f}" for c in stats_table.columns if c != "N pointings"})
stats_table.style.format(fmt)

In [ ]:
def save_fig(fig, stem):
    # Each figure is saved as .png and .pdf in FIG_DIR
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_{stem}.{ext}"), dpi=150, bbox_inches="tight")


def grid_axes(n, ncols=3, sharex=True):
    # Grid of n panels (3 columns); the extra panels are hidden
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 4.2 * nrows), sharex=sharex, squeeze=False)
    axes = axes.ravel()
    for ax in axes[n:]:
        ax.axis("off")
    return fig, axes


def plot_pointing_hists(stacked):
    # One panel per category; one histogram per filter, with the colours of the filters
    n = len(CATEGORIES)
    fig, axes = grid_axes(n, sharex=False)
    colors = [BAND_COLORS[b] for b in BANDS]
    for ax, cat in zip(axes, CATEGORIES):
        v = vpp[cat]
        vmax = int(v.max())
        # Same bins for the six filters; integer-centred bins when the counts are small
        if vmax < N_BINS:
            bins = np.arange(-0.5, vmax + 1.5)
        else:
            bins = np.linspace(0, vmax, N_BINS + 1)
        labels = [
            f"{b}: mean={v[:, i].mean():.1f}, median={np.median(v[:, i]):.1f}" for i, b in enumerate(BANDS)
        ]
        if stacked:
            ax.hist(
                [v[:, i] for i in range(len(BANDS))],
                bins=bins,
                stacked=True,
                color=colors,
                label=labels,
                edgecolor="none",
            )
        else:
            for i in range(len(BANDS)):
                # filled histogram (transparent) + outline, the legend uses the outline
                ax.hist(v[:, i], bins=bins, histtype="stepfilled", color=colors[i], alpha=0.30)
                ax.hist(v[:, i], bins=bins, histtype="step", color=colors[i], lw=1.8, label=labels[i])
        ax.set_title(cat)
        ax.grid(alpha=0.3)
        if LOG_Y:
            ax.set_yscale("log")
        ax.legend(
            title=f"visits per pointing ({len(v):,} pointings)",
            fontsize=10,
            title_fontsize=10,
            loc="upper right",
            framealpha=0.85,
        )
    for k, ax in enumerate(axes[:n]):
        if k % 3 == 0:
            ax.set_ylabel("Number of pointings")
        if k + 3 >= n:
            ax.set_xlabel("Number of visits per pointing")
    kind = "stacked" if stacked else "non-stacked"
    fig.suptitle(f"Visits per pointing, {kind} histograms - {opsim_name}\n(pointing = {POINTING_DESC})")
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    return fig

## 4. Histograms of the number of visits per pointing

### Figure 1 - Non-stacked histograms (`<run>_visits_per_pointing_hist.png`)

One panel per category, one histogram per filter (same colour code as in `10_FilterBalance`). Each pointing contributes once to the histogram of each filter. The legend gives the **mean** and the **median** number of visits per pointing in each filter, and the number of pointings of the category. Each panel has its own horizontal scale; the bins are the same for the six filters of a panel.

In [ ]:
fig1 = plot_pointing_hists(stacked=False)
save_fig(fig1, "visits_per_pointing_hist")

### Figure 2 - Stacked histograms (`<run>_visits_per_pointing_hist_stacked.png`)

Same panels and same bins as Figure 1, but the histograms of the six filters are **stacked** (order u, g, r, i, z, y from the bottom): in each bin, the total height is the number of pointings, whatever the filter, with a given number of visits, and the coloured segments give the contribution of each filter. The legend is the same as in Figure 1 (mean and median per filter).

In [ ]:
fig2 = plot_pointing_hists(stacked=True)
save_fig(fig2, "visits_per_pointing_hist_stacked")

## 5. Reading the results

- **WFD and WFD low dust**: the histograms of the filters should be peaked at different values (many more visits in `r`, `i`, `z`, `y` than in `u`, `g`); the width of each peak shows the uniformity of the depth over the footprint (a rolling cadence or the edges of the footprint broaden it).
- **DDF**: very few pointings, each with a large number of visits; the histograms are sparse and the mean and median can differ strongly from one DDF to another (see `04_DDF_NVisitsVsTime`).
- **Mini-surveys and micro-surveys**: some filters may be absent, which gives a histogram entirely in the first bin (0 visit) and a mean and a median equal to 0.
- **Mean versus median**: a mean much larger than the median indicates a long tail (a few pointings with many more visits, for instance the overlaps between regions); the pointings with 0 visit in a filter pull both down.
- **Possible extensions**: switch to `POINTING_MODE = "radec"` to count the visits per field centre; change `NSIDE_POINTING`; set `LOG_Y = True` to see the tails.